In [1]:
import pandas as pd

# Presidential Election Data

## Data SetUp

In [2]:
# Read presidential election data from 2000-2024
file_path = "/Users/bill_zyx/jupyter-1.0.0/Trust/Data/countypres_2000-2024.csv"
df_pres = pd.read_csv(file_path, low_memory=False)

In [3]:
# Column Dropping
columns_to_drop = ['state','county_name','office','candidate','version','mode']
df_pres = df_pres.drop(columns=columns_to_drop)   #Drop non-useful columns
df_pres = df_pres[df_pres['party'].isin(['DEMOCRAT', 'REPUBLICAN'])]   #Drop non-major parties
df_pres = df_pres[df_pres['totalvotes'] > 0]   #Drop problematic rows

In [4]:
# Rename columns for consistency
df_pres = df_pres.rename(
    columns={
        'state_po':'state_code',
        'county_fips':'state_county_code',
        'candidatevotes': 'candidate_votes', 
        'totalvotes':'total_votes'
    }
)

# Unify code
df_pres["state_county_code"] = (
    df_pres["state_county_code"]
        .astype("Int64")        
        .astype("string")
        .str.zfill(5)
)

## Map to District Level

In [5]:
# Import crosswalk data
file_path = "data/intermediate/county_to_cd_crosswalk.csv"
df_crosswalk = pd.read_csv(
    file_path,
    dtype={
        'state_county_code':'string',
        'district_code':'string'
    },
    low_memory=False
)

In [6]:
# Map for cd and years
cd_year_map = [
    (102, 1990, 1991),
    (103, 1992, 1998),
    (106, 1999, 2002),
    (108, 2003, 2004),
    (109, 2005, 2008),
    (111, 2009, 2012),
    (113, 2013, 2014),
    (114, 2015, 2016),
    (115, 2017, 2018),
    (116, 2019, 2020),
    (117, 2021, 2022),
    (118, 2023, 2024),
    (119, 2025, 2026),
]

In [7]:
# Assign cd number to years
def year_to_cd(year: int) -> int:
    for cd, start, end in cd_year_map:
        if start <= year <= end:
            return cd
    return pd.NA

df_pres["cd"] = df_pres["year"].apply(year_to_cd).astype("Int64")

In [8]:
# Merge election data and crosswalk
df_pres_xw = df_pres.merge(
    df_crosswalk,
    on=["state_county_code", "cd"],
    how="left"
)


In [9]:
# Calculate weighted votes
df_pres_xw["votes_w"] = df_pres_xw["candidate_votes"] * df_pres_xw["afact"]
df_pres_xw["total_votes_w"] = df_pres_xw["total_votes"] * df_pres_xw["afact"]

In [10]:
# Collapse to district level
df_pres = (
    df_pres_xw
    .groupby(
        ["year", "cd", "state_code", "district_code", "party"],
        as_index=False
    )
    .agg(
        candidate_votes=("votes_w", "sum"),
        total_votes=("total_votes_w", "sum")
    )
)

In [11]:
# Find share of votes
df_pres['vote_share'] = df_pres['candidate_votes']/df_pres['total_votes']

# Find winning party
df_pres['winner'] = df_pres.groupby(['year', 'state_code', 'district_code'])['vote_share'].transform(
    lambda x: (x == x.max()).astype(int)
)

# Define democrat winning margin
df_pres['dem_share'] = df_pres['candidate_votes'].where(df_pres['party']=='DEMOCRAT')
df_pres['dem_share'] = df_pres.groupby(['year', 'state_code', 'district_code'])['dem_share'].transform('max') / df_pres['total_votes']
df_pres['dem_win_margin'] = df_pres['dem_share'] - 0.5   # win margin relative to 50%

In [12]:
# Drop losing rows
df_pres = df_pres[~df_pres["winner"].isin([0])]
df_pres = df_pres.drop(columns='winner')

# Drop unhelpful columns
df_pres = df_pres.drop(columns='candidate_votes')
df_pres = df_pres.drop(columns='cd')

# Drop NaN values (eg., some dem_win_margin missing due to original data deficiency)
df_pres = df_pres.dropna()

In [13]:
# Export cleaned presidential election data
df_pres.to_csv("data/intermediate/presidential_election_district_level.csv")